# Encode Immune Human Dataset with scGPT

Encode the `Immune_ALL_human.h5ad` dataset using the pre-trained scGPT model
and stream embeddings to `data/immune_human_embeddings.h5ad`.

In [ ]:
from pathlib import Path

import scanpy as sc
import torch

from scfm_utils.scgpt import (
    create_scgpt_dataset,
    encode_scgpt_embeddings_to_h5ad,
    load_cls_embeddings,
    load_average_gene_embeddings,
    load_scgpt_model,
)

## Configuration

In [ ]:
DATA_PATH = Path("../data/Immune_ALL_human.h5ad")
MODEL_DIR = Path("../models/scGPT_bc")
OUTPUT_PATH = Path("../data/immune_human_embeddings.h5ad")
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Output: {OUTPUT_PATH}")

## Load model and data

In [ ]:
bundle = load_scgpt_model(MODEL_DIR, device=DEVICE)
print(f"Model: {bundle.config['embsize']}d, {bundle.config['nlayers']} layers")
print(f"Vocab: {len(bundle.vocab)} genes")

In [ ]:
adata = sc.read(DATA_PATH, cache=True)
adata.obs["celltype"] = adata.obs["final_annotation"].astype(str)
print(f"Dataset: {adata.shape[0]} cells, {adata.shape[1]} genes")
print(f"Cell types: {adata.obs['celltype'].nunique()}")

## Preprocess and create dataloader

In [ ]:
scgpt_ds = create_scgpt_dataset(
    adata,
    vocab=bundle.vocab,
    gene2idx=bundle.gene2idx,
    batch_size=BATCH_SIZE,
)
print(f"Genes in vocab: {len(scgpt_ds.genes_in_vocab)}")
print(f"Batches: {len(scgpt_ds.dataloader)}")

## Encode and stream to h5ad

In [ ]:
encode_scgpt_embeddings_to_h5ad(
    model=bundle.model,
    dataloader=scgpt_ds.dataloader,
    vocab=bundle.vocab,
    gene_names=scgpt_ds.genes_in_vocab,
    cell_types=adata.obs["celltype"].values,
    output_path=OUTPUT_PATH,
    device=DEVICE,
)
print(f"Embeddings saved to {OUTPUT_PATH}")

## Sanity check: UMAP of CLS embeddings

In [ ]:
emb_adata = load_cls_embeddings(OUTPUT_PATH)
print(f"Loaded CLS embeddings: {emb_adata.shape}")

sc.pp.neighbors(emb_adata, use_rep="X")
sc.tl.umap(emb_adata)
sc.pl.umap(emb_adata, color="celltype", title="CLS Embeddings")

## Sanity check: averaged gene embeddings per cell type

In [ ]:
import numpy as np

avgs, gene_names = load_average_gene_embeddings(OUTPUT_PATH)
print(f"Cell types: {len(avgs)}")
for ct, emb in avgs.items():
    print(f"  {ct}: {emb.shape}, mean={emb.mean():.4f}, std={emb.std():.4f}")

In [ ]:
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt

# Flatten each cell type's (n_genes, embsize) to a single vector and compute pairwise cosine distances
cell_type_names = list(avgs.keys())
flat = np.stack([avgs[ct].ravel() for ct in cell_type_names])
dist_matrix = squareform(pdist(flat, metric="cosine"))

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(dist_matrix, cmap="viridis")
ax.set_xticks(range(len(cell_type_names)))
ax.set_yticks(range(len(cell_type_names)))
ax.set_xticklabels(cell_type_names, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(cell_type_names, fontsize=8)
ax.set_title("Pairwise cosine distance between cell-type averaged gene embeddings")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print(f"Distance range: [{dist_matrix[dist_matrix > 0].min():.4f}, {dist_matrix.max():.4f}]")